# Millbrook NHS Trust — SQL Intro with DuckDB

This dataset is a synthetic acute-care **data warehouse**: 26 CSVs organised into dimension and fact tables, covering roughly 16,000 patients across three years (2023–2025). It's designed for SQL and data-analysis practice.

DuckDB is the ideal way to work with it — no database build step, no server. Point DuckDB at the CSVs and query them like any warehouse.

In this notebook we'll:

1. Register the CSVs as DuckDB views.
2. Count patients — and see why a naive `COUNT(*)` gives the wrong answer (hint: Type-2 SCD).
3. Test the 'Monday effect' in A&E arrivals.
4. Compute inpatient length-of-stay (LOS) statistics.

**Full exercise set (27 questions with hints, solutions, and discussion):** [github.com/leogodin217/nhs_sql_practice_data](https://github.com/leogodin217/nhs_sql_practice_data)

## Setup

DuckDB is lightweight and usually already present on Kaggle; install it just in case. We register each CSV as a view so we can query them by plain table names.

In [ ]:
!pip install -q duckdb

In [ ]:
import duckdb

DATA = '/kaggle/input/millbrook-nhs-acute-care-warehouse'

TABLES = [
    'dim_patient', 'dim_ward', 'dim_consultant', 'dim_procedure',
    'dim_medication', 'dim_diagnostic', 'dim_theatre',
    'fact_ed_arrival', 'fact_triage', 'fact_ed_assessment',
    'fact_admission', 'fact_ward_assignment', 'fact_medication_administered',
    'fact_icu_care', 'fact_discharge', 'fact_dtoc_assessment',
    'fact_referral_created', 'fact_appointment_attended',
    'fact_pre_op_assessment', 'fact_surgeon_assigned', 'fact_surgery_performed',
    'fact_cancer_referral', 'fact_cancer_first_seen',
    'fact_diagnostic_ordered', 'fact_diagnostic_performed',
    'fact_fft_response', 'fact_safety_incident', 'fact_death_record',
]

con = duckdb.connect()
for t in TABLES:
    con.execute(f"CREATE VIEW {t} AS SELECT * FROM read_csv_auto('{DATA}/{t}.csv')")

con.execute("SHOW TABLES").fetchdf()

## 1. How many patients does the trust serve?

`dim_patient` is a **Type-2 Slowly Changing Dimension** — when a tracked property changes, a new row is written with fresh `valid_from` / `valid_to` timestamps. A naive `COUNT(*)` counts history, not patients.

Three ways to ask the question, all useful in different contexts:

In [ ]:
con.execute("""
SELECT
    COUNT(*)                                              AS total_rows,
    COUNT(DISTINCT id)                                    AS unique_patients,
    SUM(CASE WHEN valid_to IS NULL THEN 1 ELSE 0 END)     AS current_patients
FROM dim_patient
""").fetchdf()

Expect `total_rows` to exceed `unique_patients` — the excess rows are historical SCD versions.

Always check for SCD-2 patterns before trusting row counts on dimension tables.

## 2. The Monday effect in A&E

NHS ops teams know Monday is brutal — weekend avoiders, GP referrals, and elective work restarting all converge. Is it real in this data?

We average arrivals per calendar day within each day-of-week, so three years of Mondays are compared fairly against three years of Tuesdays.

In [ ]:
con.execute("""
SELECT
    DAYNAME(timestamp)                                      AS day_of_week,
    EXTRACT(ISODOW FROM timestamp)::INT                     AS day_num,
    COUNT(*)                                                AS total_arrivals,
    COUNT(DISTINCT timestamp::DATE)                         AS calendar_days,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT timestamp::DATE), 1) AS avg_per_day
FROM fact_ed_arrival
GROUP BY day_of_week, day_num
ORDER BY day_num
""").fetchdf()

Monday should stand out noticeably above mid-week, with Saturday and Sunday clearly lower. That gap matters for rota planning — halving the staff on a Sunday looks very different from halving it on a Monday.

## 3. Inpatient length of stay

A patient's journey can have multiple `fact_admission` and `fact_discharge` rows per `spell_id`. We collapse each spell to its earliest admission timestamp and earliest discharge timestamp, then compute LOS in days.

LOS is almost always right-skewed — most stays are short, but a long tail of complex patients dominates bed-days. That's why median and p90 matter more than mean for operational planning.

In [ ]:
con.execute("""
WITH spells AS (
    SELECT
        a.spell_id,
        MIN(a.timestamp) AS admit_ts,
        MIN(d.timestamp) AS discharge_ts
    FROM fact_admission a
    JOIN fact_discharge d USING (spell_id)
    GROUP BY a.spell_id
),
los AS (
    SELECT
        spell_id,
        EXTRACT(EPOCH FROM (discharge_ts - admit_ts)) / 86400.0 AS los_days
    FROM spells
    WHERE discharge_ts > admit_ts
)
SELECT
    COUNT(*)                                    AS completed_spells,
    ROUND(AVG(los_days), 2)                     AS mean_los_days,
    ROUND(MEDIAN(los_days), 2)                  AS median_los_days,
    ROUND(QUANTILE_CONT(los_days, 0.90), 2)     AS p90_los_days,
    ROUND(MAX(los_days), 2)                     AS max_los_days
FROM los
""").fetchdf()

## Where to next

Those three queries scratch the surface. The full set of 27 exercises covers NHS performance metrics (4-hour A&E target, 62-day cancer pathway, 18-week RTT), data-quality traps, deprivation/outcome analysis, theatre utilisation, and more. Each exercise has hints, a sample solution, and a discussion section.

Repo: [github.com/leogodin217/nhs_sql_practice_data](https://github.com/leogodin217/nhs_sql_practice_data)

Have fun — and remember there's rarely a single 'right' answer. A query that reasonably answers the question is a good query.